# Georeferencing GSI Plate IV (geological map)

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Work towards georeferencing the geological map (Plate IV) of the Alwar hydrogeological atlas: reference towns from OpenStreetMap, the map frame and graticule ticks detected with OpenCV, the embedded original image, and an affine fit of the graticule. Control points are candidates until reviewed.

| | |
|---|---|
| **Reads** | `MODFLOW_Alwar/…/Alwar_Plate_IV_Geological_Map_HighRes.png`, the atlas PDF |
| **Writes** | `MODFLOW_Alwar/…` (`Step7ID4F…`, `Step7ID4G…` registers and images) |
| **Cells** | 31 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. Reference towns (OpenStreetMap)
- 2. Map frame, graticule and tick detection
- 3. Embedded original image and working crop
- 4. Edge geometry and affine fit

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["geopy", "opencv-python", "pymupdf", "pillow", "openpyxl", "geopandas"], modflow=False)
del TOKEN, URL


## 1. Reference towns (OpenStreetMap)

**[orig #299]** Set project and model folders  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

PROJECT = DRIVE / "gw_framework"
MODEL = PROJECT / "MODFLOW_Alwar"

print("Google Drive mounted:", DRIVE.exists())
print("Research folder exists:", PROJECT.exists())
print("MODFLOW folder exists:", MODEL.exists())

print("\nResearch folders:")
if PROJECT.exists():
    for p in PROJECT.iterdir():
        if p.is_dir():
            print(p.name)


**[orig #300]** Load the OSM reference-coordinate candidates and the control-point register  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from pathlib import Path
import pandas as pd

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence"
)

GCP_DIR = BASE / "Step7ID4F_Georeferencing_Preparation"

files = [
    GCP_DIR / "Step7ID4F3_OSM_Reference_Coordinates_Candidates.csv",
    GCP_DIR / "Step7ID4F3_OSM_Reference_Coordinates_Candidates.xlsx",
    GCP_DIR / "Step7ID4F2_Feature_Control_Point_Register.xlsx"
]

for f in files:
    print(f.name, "EXISTS:", f.exists())

csv_file = files[0]

if csv_file.exists():
    df = pd.read_csv(csv_file)
    print("\nCandidate coordinate records:", len(df))
    print(df[[
        "Search_Name",
        "Reference_Latitude",
        "Reference_Longitude",
        "Geocoding_Status"
    ]].to_string(index=False))

print("\nGeoreferencing folder:", GCP_DIR)


**[orig #301]** Geocode town candidates with OpenStreetMap (geopy)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

!pip -q install geopy

from pathlib import Path
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd
import time

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4F_Georeferencing_Preparation"
)

geolocator = Nominatim(
    user_agent="Alwar_PhD_Geological_Map_Research_2026"
)

search = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.2,
    swallow_exceptions=True
)

places = [
    "Alwar town, Rajasthan, India",
    "Bansur town, Rajasthan, India",
    "Behror town, Rajasthan, India",
    "Tijara town, Rajasthan, India",
    "Kathumar town, Rajasthan, India",
    "Kishangarh Bas town, Rajasthan, India",
    "Ramgarh town, Alwar, Rajasthan, India",
    "Rajgarh town, Alwar, Rajasthan, India",
    "Thanagazi town, Rajasthan, India",
    "Mundawar town, Rajasthan, India"
]

records = []

for query in places:
    print("\nSearching:", query)

    results = search(
        query,
        exactly_one=False,
        limit=5,
        country_codes="in",
        addressdetails=True
    )

    if not results:
        records.append({
            "Search_Query": query,
            "Candidate_Rank": None,
            "Display_Name": None,
            "Latitude": None,
            "Longitude": None,
            "OSM_Class": None,
            "OSM_Type": None,
            "Address_Type": None,
            "OSM_ID": None,
            "Review_Status": "NO_RESULT"
        })
        print("  No result")
        continue

    for rank, result in enumerate(results, start=1):
        raw = result.raw

        records.append({
            "Search_Query": query,
            "Candidate_Rank": rank,
            "Display_Name": result.address,
            "Latitude": result.latitude,
            "Longitude": result.longitude,
            "OSM_Class": raw.get("class"),
            "OSM_Type": raw.get("type"),
            "Address_Type": raw.get("addresstype"),
            "OSM_ID": raw.get("osm_id"),
            "Review_Status": "CANDIDATE_NOT_VERIFIED"
        })

        print(
            rank,
            raw.get("class"),
            raw.get("type"),
            result.latitude,
            result.longitude,
            result.address
        )

df = pd.DataFrame(records)

csv_path = BASE / "Step7ID4F5_Refined_OSM_Town_Candidates.csv"
xlsx_path = BASE / "Step7ID4F5_Refined_OSM_Town_Candidates.xlsx"

df.to_csv(csv_path, index=False)
df.to_excel(xlsx_path, index=False)

print("\nSTEP 7I-D.4F.5 COMPLETED")
print("Total candidate results:", len(df))
print("Distinct search queries:", len(places))
print("Candidate coordinates:", df["Latitude"].notna().sum())
print("Accepted GCPs: 0")

print("\nSaved:")
print(csv_path)
print(xlsx_path)


**[orig #302]** Town candidate spatial register and GeoJSON (candidates remain provisional)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import json

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4F_Georeferencing_Preparation"
)

SOURCE = BASE / "Step7ID4F5_Refined_OSM_Town_Candidates.csv"

df = pd.read_csv(SOURCE)

# Retain only the six returned town candidates.
valid = df.dropna(
    subset=["Latitude", "Longitude"]
).copy()

valid = valid.sort_values("Search_Query").reset_index(drop=True)

# Candidate status remains provisional.
valid["GCP_Status"] = "CANDIDATE_NOT_VERIFIED"
valid["Map_Pixel_X"] = pd.NA
valid["Map_Pixel_Y"] = pd.NA
valid["Accepted_GCP"] = "NO"

# Save clean candidate register.
csv_out = BASE / "Step7ID4F6_Town_Candidate_Spatial_Register.csv"
valid.to_csv(csv_out, index=False)

# Create GeoJSON for QGIS inspection.
features = []

for _, row in valid.iterrows():
    features.append({
        "type": "Feature",
        "properties": {
            "place": row["Search_Query"],
            "osm_display_name": row["Display_Name"],
            "osm_class": row["OSM_Class"],
            "osm_type": row["OSM_Type"],
            "status": "CANDIDATE_NOT_VERIFIED",
            "accepted_gcp": False
        },
        "geometry": {
            "type": "Point",
            "coordinates": [
                float(row["Longitude"]),
                float(row["Latitude"])
            ]
        }
    })

geojson = {
    "type": "FeatureCollection",
    "name": "Alwar_OSM_Town_Candidates",
    "features": features
}

geojson_out = BASE / "Step7ID4F6_Town_Candidate_Points.geojson"

with open(geojson_out, "w", encoding="utf-8") as f:
    json.dump(geojson, f, indent=2)

# Plot geographic distribution.
fig, ax = plt.subplots(figsize=(9, 9))

ax.scatter(
    valid["Longitude"],
    valid["Latitude"],
    s=65,
    marker="o"
)

for _, row in valid.iterrows():
    name = row["Search_Query"].split(",")[0].replace(" town", "")
    ax.annotate(
        name,
        (row["Longitude"], row["Latitude"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9
    )

ax.set_xlabel("Longitude (degrees)")
ax.set_ylabel("Latitude (degrees)")
ax.set_title(
    "Provisional OSM Town Coordinates – Alwar Region\n"
    "Not accepted as geological-map GCPs"
)
ax.grid(True)
ax.set_aspect("equal", adjustable="datalim")

plt.tight_layout()

png_out = BASE / "Step7ID4F6_Town_Candidate_Distribution.png"
plt.savefig(png_out, dpi=300, bbox_inches="tight")
plt.show()

print("\nSTEP 7I-D.4F.6 COMPLETED")
print("Candidate towns:", len(valid))
print("Accepted GCPs:", 0)

print("\nCandidate locations:")
print(valid[[
    "Search_Query",
    "Latitude",
    "Longitude",
    "OSM_Type",
    "GCP_Status"
]].to_string(index=False))

print("\nSaved files:")
print(csv_out)
print(geojson_out)
print(png_out)


## 2. Map frame, graticule and tick detection

**[orig #303]** Crop Plate IV with its graticule from the high-resolution map  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from PIL import Image
from IPython.display import display

# Input: original high-resolution geological map
BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map"
)

IMG_PATH = BASE / "Alwar_Plate_IV_Geological_Map_HighRes.png"

# Output folder
OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)
OUT.mkdir(parents=True, exist_ok=True)

assert IMG_PATH.exists(), f"Image not found: {IMG_PATH}"

img = Image.open(IMG_PATH).convert("RGB")
W, H = img.size

print("Original image:", img.size)

# Crop Plate IV only, retaining its coordinate frame and margins.
# Fractions are relative to the complete atlas page.
plate = img.crop((
    int(W * 0.09),
    int(H * 0.10),
    int(W * 0.485),
    int(H * 0.635)
))

plate_path = OUT / "Step7ID4G1_Plate_IV_With_Graticule.png"
plate.save(plate_path)

# Extract individual margin strips for coordinate reading
pw, ph = plate.size

margins = {
    "TOP": (0, 0, pw, int(ph * 0.12)),
    "BOTTOM": (0, int(ph * 0.87), pw, ph),
    "LEFT": (0, 0, int(pw * 0.10), ph),
    "RIGHT": (int(pw * 0.90), 0, pw, ph)
}

for name, bounds in margins.items():
    crop = plate.crop(bounds)
    crop.save(OUT / f"Step7ID4G1_{name}_Coordinate_Margin.png")
    print(name, "crop size:", crop.size)

print("\nPlate IV crop:", plate.size)
print("Saved to:", OUT)

# Display the complete Plate IV crop
preview = plate.copy()
preview.thumbnail((1600, 1600))
display(preview)

# Display coordinate margins at enlarged size
for name in ["TOP", "BOTTOM", "LEFT", "RIGHT"]:
    print("\n", name, "COORDINATE MARGIN")
    margin = Image.open(
        OUT / f"Step7ID4G1_{name}_Coordinate_Margin.png"
    )
    display(margin.resize((margin.width * 2, margin.height * 2)))


**[orig #304]** Graticule coordinate register  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import pandas as pd
from pathlib import Path

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)
OUT.mkdir(parents=True, exist_ok=True)

records = [
    ["L1", "LEFT", "LATITUDE", "28°18'09\"", 28 + 18/60 + 9/3600],
    ["L2", "LEFT", "LATITUDE", "28°00'00\"", 28.0],
    ["L3", "LEFT", "LATITUDE", "27°30'00\"", 27.5],
    ["L4", "LEFT", "LATITUDE", "26°59'47\"", 26 + 59/60 + 47/3600],

    ["B1", "BOTTOM", "LONGITUDE", "75°46'03\"", 75 + 46/60 + 3/3600],
    ["B2", "BOTTOM", "LONGITUDE", "76°00'00\"", 76.0],
    ["B3", "BOTTOM", "LONGITUDE", "76°30'00\"", 76.5],
    ["B4", "BOTTOM", "LONGITUDE", "77°00'00\"", 77.0],
    ["B5", "BOTTOM", "LONGITUDE", "77°23'02\"", 77 + 23/60 + 2/3600],

    ["T1", "TOP", "LONGITUDE", "75°46'36\"", 75 + 46/60 + 36/3600],
    ["T2", "TOP", "LONGITUDE", "76°00'00\"", 76.0],
    ["T3", "TOP", "LONGITUDE", "76°30'00\"", 76.5],
    ["T4", "TOP", "LONGITUDE", "77°00'00\"", 77.0],
    ["T5", "TOP", "LONGITUDE", "77°24'45\"", 77 + 24/60 + 45/3600],
]

df = pd.DataFrame(records, columns=[
    "Point_ID", "Map_Margin", "Coordinate_Type",
    "Printed_DMS", "Coordinate_Decimal"
])

df["Pixel_X"] = pd.NA
df["Pixel_Y"] = pd.NA
df["Pixel_Position_Status"] = "PENDING_PIXEL_MEASUREMENT"
df["Coordinate_Status"] = "TRANSCRIBED_PROVISIONAL"

csv_path = OUT / "Step7ID4G2_Graticule_Coordinate_Register.csv"
xlsx_path = OUT / "Step7ID4G2_Graticule_Coordinate_Register.xlsx"

df.to_csv(csv_path, index=False)
df.to_excel(xlsx_path, index=False)

print("Graticule points:", len(df))
print("Latitude points:", (df.Coordinate_Type == "LATITUDE").sum())
print("Longitude points:", (df.Coordinate_Type == "LONGITUDE").sum())
print("Pixel positions measured:", df.Pixel_X.notna().sum())
print("Saved:", csv_path)
print("Saved:", xlsx_path)
display(df)


**[orig #305]** Draw a 100-pixel grid on the Plate IV crop  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

img_path = BASE / "Alwar_Plate_IV_Geological_Map_HighRes.png"
img = Image.open(img_path).convert("RGB")
W, H = img.size

# Same Plate IV crop used in the previous step
x0, y0 = int(W*.09), int(H*.10)
x1, y1 = int(W*.485), int(H*.635)
plate = img.crop((x0, y0, x1, y1))

draw = ImageDraw.Draw(plate)

# Draw a 100-pixel grid.
step = 100
for x in range(0, plate.width, step):
    draw.line((x, 0, x, plate.height), fill=(255, 0, 0), width=1)
    draw.text((x+3, 5), str(x+x0), fill=(255, 0, 0),
              stroke_width=1, stroke_fill="white")

for y in range(0, plate.height, step):
    draw.line((0, y, plate.width, y), fill=(255, 0, 0), width=1)
    draw.text((5, y+3), str(y+y0), fill=(255, 0, 0),
              stroke_width=1, stroke_fill="white")

grid_path = OUT / "Step7ID4G3_Plate_IV_Original_Pixel_Grid.png"
plate.save(grid_path)

print("Original image dimensions:", img.size)
print("Crop origin in original image:", (x0, y0))
print("Grid image dimensions:", plate.size)
print("Saved:", grid_path)

# Display a resized preview
preview = plate.copy()
preview.thumbnail((1500, 1500))
display(preview)


**[orig #306]** Inspect the approximate neatline bounds  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from PIL import Image
from pathlib import Path
from IPython.display import display

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

img = Image.open(
    BASE / "Alwar_Plate_IV_Geological_Map_HighRes.png"
).convert("RGB")

W, H = img.size
print("Original dimensions:", W, H)

# Plate IV approximate neatline bounds in original image pixels.
# These are crop locations, NOT accepted GCP coordinates.
x_left  = int(W * 0.098)
x_right = int(W * 0.477)
y_top   = int(H * 0.105)
y_bottom= int(H * 0.635)

regions = {
    "LEFT": (x_left-45, y_top, x_left+100, y_bottom),
    "TOP": (x_left, y_top-45, x_right, y_top+100),
    "BOTTOM": (x_left, y_bottom-100, x_right, y_bottom+45),
    "RIGHT": (x_right-100, y_top, x_right+45, y_bottom)
}

for name, box in regions.items():
    crop = img.crop(box)
    crop = crop.resize((crop.width*3, crop.height*3))
    path = OUT / f"Step7ID4G4_{name}_Tick_Inspection.png"
    crop.save(path)
    print("\n", name, "inspection crop:", crop.size)
    display(crop)

print("\nInspection images saved in:", OUT)


**[orig #307]** Summarise the graticule observations  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from pathlib import Path
import pandas as pd

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

csv_path = OUT / "Step7ID4G2_Graticule_Coordinate_Register.csv"

df = pd.read_csv(csv_path)

right_points = [
    ["R1", "RIGHT", "LATITUDE", "28°17'01\"", 28 + 17/60 + 1/3600],
    ["R2", "RIGHT", "LATITUDE", "28°00'00\"", 28.0],
    ["R3", "RIGHT", "LATITUDE", "27°30'00\"", 27.5],
    ["R4", "RIGHT", "LATITUDE", "26°58'42\"", 26 + 58/60 + 42/3600],
]

extra = pd.DataFrame(right_points, columns=[
    "Point_ID", "Map_Margin", "Coordinate_Type",
    "Printed_DMS", "Coordinate_Decimal"
])

extra["Pixel_X"] = pd.NA
extra["Pixel_Y"] = pd.NA
extra["Pixel_Position_Status"] = "PENDING_PIXEL_MEASUREMENT"
extra["Coordinate_Status"] = "TRANSCRIBED_PROVISIONAL"

df = pd.concat([df, extra], ignore_index=True)
df = df.drop_duplicates(subset=["Point_ID"], keep="last")

df.to_csv(csv_path, index=False)
df.to_excel(
    OUT / "Step7ID4G2_Graticule_Coordinate_Register.xlsx",
    index=False
)

print("Total graticule observations:", len(df))
print("Latitude observations:", (df.Coordinate_Type == "LATITUDE").sum())
print("Longitude observations:", (df.Coordinate_Type == "LONGITUDE").sum())
print("Pixel positions measured:", df.Pixel_X.notna().sum())
display(df)


**[orig #308]** Cut four measurement strips around the neatline  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from PIL import Image, ImageDraw
from pathlib import Path
from IPython.display import display

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

img = Image.open(
    BASE / "Alwar_Plate_IV_Geological_Map_HighRes.png"
).convert("RGB")

W, H = img.size
print("Original image dimensions:", W, H)

# Approximate Plate IV frame bounds.
# These are crop limits, not measured control-point coordinates.
xL, xR = int(W*.098), int(W*.477)
yT, yB = int(H*.105), int(H*.635)

# Narrow strips around the map neatline
regions = {
    "LEFT": (xL-65, yT-20, xL+45, yB+20),
    "RIGHT": (xR-45, yT-20, xR+65, yB+20),
    "TOP": (xL-20, yT-65, xR+20, yT+45),
    "BOTTOM": (xL-20, yB-45, xR+20, yB+65)
}

for name, bounds in regions.items():
    crop = img.crop(bounds)
    ox, oy = bounds[0], bounds[1]

    draw = ImageDraw.Draw(crop)

    # Original-pixel coordinate grid at 25-pixel intervals
    for gx in range((ox//25)*25, bounds[2]+1, 25):
        px = gx - ox
        draw.line((px, 0, px, crop.height), fill=(255, 0, 0), width=1)
        if px < crop.width-30:
            draw.text((px+2, 2), str(gx), fill=(255, 0, 0),
                      stroke_width=1, stroke_fill="white")

    for gy in range((oy//25)*25, bounds[3]+1, 25):
        py = gy - oy
        draw.line((0, py, crop.width, py), fill=(255, 0, 0), width=1)
        if py < crop.height-25:
            draw.text((2, py+2), str(gy), fill=(255, 0, 0),
                      stroke_width=1, stroke_fill="white")

    # Enlarge for visual measurement
    enlarged = crop.resize((crop.width*4, crop.height*4))
    path = OUT / f"Step7ID4G5_{name}_Pixel_Measurement.png"
    enlarged.save(path)

    print(name, "saved:", path)
    display(enlarged)

print("All four measurement strips created.")


**[orig #309]** Bottom neatline and longitude labels  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from PIL import Image
from pathlib import Path
from IPython.display import display

p = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map/"
    "Alwar_Plate_IV_Geological_Map_HighRes.png"
)

img = Image.open(p)
W, H = img.size

print("Original dimensions:", W, H)

# Bottom neatline and longitude labels
crop = img.crop((
    int(W * 0.09),
    int(H * 0.57),
    int(W * 0.49),
    int(H * 0.65)
))

display(crop.resize((crop.width * 3, crop.height * 3)))

**[orig #310]** Map-frame region with pixel axes  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)
OUT.mkdir(parents=True, exist_ok=True)

path = BASE / "Alwar_Plate_IV_Geological_Map_HighRes.png"
assert path.exists(), f"Missing image: {path}"

img = Image.open(path).convert("RGB")
W, H = img.size

print("Original image dimensions:", W, "×", H)
print("Pixel origin: top-left (0, 0)")
print("X increases rightward; Y increases downward.")

# Approximate Plate IV map-frame region, scaled to actual image dimensions.
# These are inspection boundaries, not control-point measurements.
xL = int(W * 0.075)
xR = int(W * 0.485)
yT = int(H * 0.065)
yB = int(H * 0.600)

regions = {
    "LEFT":   (xL-35, yT-10, xL+75, yB+10),
    "RIGHT":  (xR-75, yT-10, xR+35, yB+10),
    "TOP":    (xL-10, yT-35, xR+10, yT+75),
    "BOTTOM": (xL-10, yB-75, xR+10, yB+35)
}

for name, (x0, y0, x1, y1) in regions.items():
    x0, y0 = max(0, x0), max(0, y0)
    x1, y1 = min(W, x1), min(H, y1)

    crop = img.crop((x0, y0, x1, y1))
    arr = np.asarray(crop)

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(
        arr,
        extent=(x0, x1, y1, y0),
        interpolation="nearest"
    )
    ax.set_title(f"{name} margin — original image pixel coordinates")
    ax.set_xlabel("Original image X (pixels)")
    ax.set_ylabel("Original image Y (pixels)")
    ax.set_xticks(np.arange(x0, x1+1, 25))
    ax.set_yticks(np.arange(y0, y1+1, 25))
    ax.grid(True, linewidth=0.5)
    plt.xticks(rotation=90)
    plt.tight_layout()

    output = OUT / f"Step7ID4G6_{name}_Original_Pixel_Axes.png"
    fig.savefig(output, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(name, "crop bounds:", (x0, y0, x1, y1))
    print("Saved:", output)

print("\nStep 7I-D.4G.6 inspection crops completed.")


**[orig #311]** Left neatline and latitude labels  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

img = Image.open(
    BASE / "Alwar_Plate_IV_Geological_Map_HighRes.png"
).convert("RGB")

W, H = img.size

# Left neatline and latitude labels
x0 = int(W * 0.065)
x1 = int(W * 0.155)
y0 = int(H * 0.07)
y1 = int(H * 0.61)

crop = img.crop((x0, y0, x1, y1))

fig, ax = plt.subplots(figsize=(9, 14))
ax.imshow(crop, extent=(x0, x1, y1, y0),
          interpolation="nearest")

ax.set_title("LEFT margin — original pixel coordinates")
ax.set_xlabel("Original image X (pixels)")
ax.set_ylabel("Original image Y (pixels)")
ax.set_xticks(range(x0, x1+1, 10))
ax.set_yticks(range(y0, y1+1, 25))
ax.grid(True, linewidth=0.5)

plt.tight_layout()

out = OUT / "Step7ID4G6_LEFT_Clear_Pixel_Axes.png"
plt.savefig(out, dpi=250, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", out)


**[orig #312]** Detect dark frame lines (OpenCV, first version)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import cv2
import numpy as np
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map"
)

img_path = BASE / "Alwar_Plate_IV_Geological_Map_HighRes.png"

img = cv2.imread(str(img_path))
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

H, W = gray.shape
print("Original image:", W, "x", H)

# Inspect approximate Plate IV map-frame region
x0, x1 = int(W*.07), int(W*.49)
y0, y1 = int(H*.06), int(H*.62)

crop = gray[y0:y1, x0:x1]

# Detect dark frame lines
edges = cv2.Canny(crop, 50, 150)

lines = cv2.HoughLinesP(
    edges,
    rho=1,
    theta=np.pi/180,
    threshold=100,
    minLineLength=250,
    maxLineGap=15
)

results = []

if lines is not None:
    for line in lines[:, 0]:
        xA, yA, xB, yB = map(int, line)

        # Convert crop coordinates to original image coordinates
        xA += x0
        xB += x0
        yA += y0
        yB += y0

        length = np.hypot(xB-xA, yB-yA)
        angle = np.degrees(np.arctan2(yB-yA, xB-xA))

        results.append((xA, yA, xB, yB, round(length, 1),
                        round(angle, 2)))

print("Detected line segments:", len(results))

for r in sorted(results, key=lambda z: z[4], reverse=True)[:30]:
    print(r)


**[orig #313]** Detect frame line segments (robust version) → CSV  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import cv2
import numpy as np
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)
OUT.mkdir(parents=True, exist_ok=True)

img_path = BASE / "Alwar_Plate_IV_Geological_Map_HighRes.png"

img = cv2.imread(str(img_path))
assert img is not None, "Could not read original map image."

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
H, W = gray.shape

print("Original image:", W, "x", H)

# Approximate Plate IV inspection region
x0, x1 = int(W*.07), int(W*.49)
y0, y1 = int(H*.06), int(H*.62)

crop = gray[y0:y1, x0:x1]
edges = cv2.Canny(crop, 50, 150)

lines = cv2.HoughLinesP(
    edges,
    rho=1,
    theta=np.pi/180,
    threshold=100,
    minLineLength=250,
    maxLineGap=15
)

results = []

if lines is not None:
    # Robust handling of OpenCV output shapes
    line_array = np.asarray(lines).reshape(-1, 4)

    for line in line_array:
        xA, yA, xB, yB = [int(v) for v in line]

        # Convert to original-image coordinates
        xA += x0
        xB += x0
        yA += y0
        yB += y0

        length = float(np.hypot(xB-xA, yB-yA))
        angle = float(np.degrees(np.arctan2(yB-yA, xB-xA)))

        orientation = (
            "HORIZONTAL" if abs(angle) < 10
            else "VERTICAL" if abs(angle) > 80
            else "DIAGONAL"
        )

        results.append({
            "X1": xA, "Y1": yA,
            "X2": xB, "Y2": yB,
            "Length_pixels": round(length, 2),
            "Angle_degrees": round(angle, 2),
            "Orientation": orientation
        })

print("Detected line segments:", len(results))

if results:
    import pandas as pd

    df = pd.DataFrame(results)
    df = df.sort_values("Length_pixels", ascending=False)

    display(df.head(30))

    csv_path = OUT / "Step7ID4G6_Detected_Frame_Line_Segments.csv"
    df.to_csv(csv_path, index=False)

    print("Saved:", csv_path)
else:
    print("No line segments detected. We will adjust detection parameters.")


**[orig #314]** Draw the two candidate frame outlines  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import cv2
import pandas as pd
from pathlib import Path
from IPython.display import display, Image as IPyImage

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

img_path = BASE / "Alwar_Plate_IV_Geological_Map_HighRes.png"
csv_path = OUT / "Step7ID4G6_Detected_Frame_Line_Segments.csv"

img = cv2.imread(str(img_path))
df = pd.read_csv(csv_path)

# Original image dimensions
H, W = img.shape[:2]

# Draw the two candidate outlines.
# These are visual inspection candidates, not accepted boundaries.
annotated = img.copy()

# Outer outline
cv2.rectangle(
    annotated, (333, 236), (2061, 1843),
    (0, 0, 255), 3
)

# Inner outline
cv2.rectangle(
    annotated, (413, 317), (2019, 1761),
    (0, 180, 0), 3
)

cv2.putText(
    annotated, "OUTER CANDIDATE",
    (350, 225), cv2.FONT_HERSHEY_SIMPLEX,
    0.8, (0, 0, 255), 2
)

cv2.putText(
    annotated, "INNER CANDIDATE",
    (430, 305), cv2.FONT_HERSHEY_SIMPLEX,
    0.8, (0, 180, 0), 2
)

# Crop around the two outlines
crop = annotated[180:1900, 280:2110]

output = OUT / "Step7ID4G7_Frame_Outline_Inspection.png"
cv2.imwrite(str(output), crop)

print("Saved:", output)
display(IPyImage(filename=str(output)))

print("\nCandidate outline dimensions:")
print("Outer width:", 2061-333, "pixels")
print("Outer height:", 1843-236, "pixels")
print("Inner width:", 2019-413, "pixels")
print("Inner height:", 1761-317, "pixels")


**[orig #315]** Corner control-point candidates  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import pandas as pd
from pathlib import Path

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)
OUT.mkdir(parents=True, exist_ok=True)

corners = [
    ["GCP-C1", "TOP_LEFT", 413, 317,
     "28°18'09\"", "75°46'36\""],
    ["GCP-C2", "TOP_RIGHT", 2019, 317,
     "28°17'01\"", "77°24'45\""],
    ["GCP-C3", "BOTTOM_LEFT", 413, 1761,
     "26°59'47\"", "75°46'03\""],
    ["GCP-C4", "BOTTOM_RIGHT", 2019, 1761,
     "26°58'42\"", "77°23'02\""]
]

df = pd.DataFrame(corners, columns=[
    "Point_ID", "Corner",
    "Pixel_X_Approx", "Pixel_Y_Approx",
    "Latitude_DMS", "Longitude_DMS"
])

def dms_to_decimal(value):
    import re
    nums = [float(x) for x in re.findall(r"\d+", value)]
    return nums[0] + nums[1]/60 + nums[2]/3600

df["Latitude_DD"] = df["Latitude_DMS"].apply(dms_to_decimal)
df["Longitude_DD"] = df["Longitude_DMS"].apply(dms_to_decimal)

df["Pixel_Status"] = "FRAME_LINE_CANDIDATE_NOT_VERIFIED"
df["GCP_Status"] = "PENDING_CORNER_LABEL_VERIFICATION"

csv = OUT / "Step7ID4G8_Corner_Control_Point_Candidates.csv"
xlsx = OUT / "Step7ID4G8_Corner_Control_Point_Candidates.xlsx"

df.to_csv(csv, index=False)
df.to_excel(xlsx, index=False)

print("Corner candidates:", len(df))
print("Accepted GCPs: 0")
print("Saved:", csv)
print("Saved:", xlsx)

display(df)


**[orig #316]** Interior graticule register  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from pathlib import Path
import pandas as pd

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

points = [
    ["L2", "LEFT", "LATITUDE", "28°00'00\"", 28.0],
    ["L3", "LEFT", "LATITUDE", "27°30'00\"", 27.5],
    ["R2", "RIGHT", "LATITUDE", "28°00'00\"", 28.0],
    ["R3", "RIGHT", "LATITUDE", "27°30'00\"", 27.5],
    ["T2", "TOP", "LONGITUDE", "76°00'00\"", 76.0],
    ["T3", "TOP", "LONGITUDE", "76°30'00\"", 76.5],
    ["T4", "TOP", "LONGITUDE", "77°00'00\"", 77.0],
    ["B2", "BOTTOM", "LONGITUDE", "76°00'00\"", 76.0],
    ["B3", "BOTTOM", "LONGITUDE", "76°30'00\"", 76.5],
    ["B4", "BOTTOM", "LONGITUDE", "77°00'00\"", 77.0],
]

df = pd.DataFrame(points, columns=[
    "Point_ID", "Margin", "Coordinate_Type",
    "Printed_DMS", "Coordinate_DD"
])

df["Pixel_X"] = pd.NA
df["Pixel_Y"] = pd.NA
df["Pixel_Status"] = "PENDING_MANUAL_MEASUREMENT"
df["GCP_Status"] = "NOT_ACCEPTED"

df.to_csv(
    OUT / "Step7ID4G9_Interior_Graticule_Register.csv",
    index=False
)
df.to_excel(
    OUT / "Step7ID4G9_Interior_Graticule_Register.xlsx",
    index=False
)

print("Interior graticule points:", len(df))
print("Measured pixel positions:", 0)
print("Accepted GCPs:", 0)
display(df)


**[orig #317]** Detect candidate tick segments near the frame  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

img_path = BASE / "Alwar_Plate_IV_Geological_Map_HighRes.png"
img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
assert img is not None, "Original map image not found."

H, W = img.shape

# Candidate map neatline bounds from previous visual inspection
xL, xR = 413, 2019
yT, yB = 317, 1761

# Search bands near each frame edge
bands = {
    "LEFT": (xL-5, yT, xL+65, yB),
    "RIGHT": (xR-65, yT, xR+5, yB),
    "TOP": (xL, yT-5, xR, yT+65),
    "BOTTOM": (xL, yB-65, xR, yB+5)
}

records = []
annotated = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

for side, (xa, ya, xb, yb) in bands.items():
    crop = img[ya:yb, xa:xb]
    edges = cv2.Canny(crop, 40, 120)

    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi/180,
        threshold=8,
        minLineLength=8,
        maxLineGap=3
    )

    if lines is None:
        continue

    for line in np.asarray(lines).reshape(-1, 4):
        a, b, c, d = [int(v) for v in line]

        X1, Y1 = a+xa, b+ya
        X2, Y2 = c+xa, d+ya

        dx, dy = X2-X1, Y2-Y1
        length = float(np.hypot(dx, dy))
        angle = abs(float(np.degrees(np.arctan2(dy, dx))))

        # Retain short, approximately perpendicular tick segments
        if not (8 <= length <= 65):
            continue

        if side in ["LEFT", "RIGHT"] and angle <= 75:
            continue

        if side in ["TOP", "BOTTOM"] and angle >= 15:
            continue

        records.append({
            "Margin": side,
            "X1": X1, "Y1": Y1,
            "X2": X2, "Y2": Y2,
            "Centre_X": round((X1+X2)/2, 2),
            "Centre_Y": round((Y1+Y2)/2, 2),
            "Length_px": round(length, 2),
            "Angle_deg": round(angle, 2)
        })

        cv2.line(
            annotated, (X1, Y1), (X2, Y2),
            (0, 0, 255), 3
        )

df = pd.DataFrame(records)

print("Candidate tick segments detected:", len(df))

if len(df):
    df = df.sort_values(["Margin", "Centre_Y", "Centre_X"])
    display(df)

    csv = OUT / "Step7ID4G10_Candidate_Tick_Segments.csv"
    df.to_csv(csv, index=False)
    print("Saved:", csv)

# Display only the map-frame region with detected candidates
preview = annotated[yT-20:yB+20, xL-20:xR+20]

plt.figure(figsize=(12, 10))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.title("Candidate graticule tick detections — red segments")
plt.axis("off")
plt.show()

preview_path = OUT / "Step7ID4G10_Tick_Detection_Overlay.png"
cv2.imwrite(str(preview_path), annotated)
print("Overlay saved:", preview_path)


**[orig #318]** Corrected tick detection (bands just inside the neatline)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

img = cv2.imread(
    str(BASE / "Alwar_Plate_IV_Geological_Map_HighRes.png")
)
assert img is not None

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Candidate neatline coordinates
xL, xR = 413, 2019
yT, yB = 317, 1761

# Narrow bands immediately inside the neatline
bands = {
    "LEFT": (xL, yT, xL+45, yB),
    "RIGHT": (xR-45, yT, xR, yB),
    "TOP": (xL, yT, xR, yT+45),
    "BOTTOM": (xL, yB-45, xR, yB)
}

records = []
overlay = img.copy()

for side, (xa, ya, xb, yb) in bands.items():

    crop = gray[ya:yb, xa:xb]
    edges = cv2.Canny(crop, 40, 120)

    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi/180,
        threshold=8,
        minLineLength=8,
        maxLineGap=3
    )

    if lines is None:
        continue

    for line in np.asarray(lines).reshape(-1, 4):
        a, b, c, d = map(int, line)

        X1, Y1 = a+xa, b+ya
        X2, Y2 = c+xa, d+ya

        dx, dy = X2-X1, Y2-Y1
        length = np.hypot(dx, dy)
        angle = abs(np.degrees(np.arctan2(dy, dx)))

        # Correct orientation:
        # LEFT/RIGHT ticks are horizontal
        # TOP/BOTTOM ticks are vertical
        if side in ("LEFT", "RIGHT"):
            valid_orientation = angle <= 15
        else:
            valid_orientation = angle >= 75

        if not valid_orientation or not (8 <= length <= 60):
            continue

        records.append({
            "Margin": side,
            "X1": X1, "Y1": Y1,
            "X2": X2, "Y2": Y2,
            "Centre_X": round((X1+X2)/2, 2),
            "Centre_Y": round((Y1+Y2)/2, 2),
            "Length_px": round(float(length), 2),
            "Angle_deg": round(float(angle), 2)
        })

        cv2.line(
            overlay, (X1, Y1), (X2, Y2),
            (0, 0, 255), 3
        )

df = pd.DataFrame(records)

print("Corrected candidate tick segments:", len(df))

if not df.empty:
    df = df.sort_values(["Margin", "Centre_Y", "Centre_X"])
    display(df)

    df.to_csv(
        OUT / "Step7ID4G11_Corrected_Tick_Candidates.csv",
        index=False
    )

preview = overlay[yT-25:yB+25, xL-25:xR+25]

plt.figure(figsize=(12, 10))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.title("Corrected candidate graticule ticks — red")
plt.axis("off")
plt.show()

cv2.imwrite(
    str(OUT / "Step7ID4G11_Corrected_Tick_Overlay.png"),
    overlay
)

print("Outputs saved:", OUT)


**[orig #319]** Consolidate tick candidates  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

src = OUT / "Step7ID4G11_Corrected_Tick_Candidates.csv"
assert src.exists(), f"Detection CSV not found: {src}"

df = pd.read_csv(src)

print("Raw detections:", len(df))
print("Detections by margin:")
print(df.groupby("Margin").size())

# Remove detections very close to the neatline corners.
# Retain only the expected interior tick search ranges.
ranges = {
    "LEFT": (450, 1350),
    "RIGHT": (450, 1350),
    "TOP": (450, 1800),
    "BOTTOM": (450, 1800)
}

records = []

for margin, group in df.groupby("Margin"):
    low, high = ranges[margin]

    # Tick position is Y on side margins, X on top/bottom.
    axis = "Centre_Y" if margin in ["LEFT", "RIGHT"] else "Centre_X"

    group = group[
        group[axis].between(low, high)
    ].sort_values(axis).copy()

    if group.empty:
        continue

    # Cluster duplicate or fragmented detections within 8 pixels.
    clusters = []
    current = [group.iloc[0]]

    for _, row in group.iloc[1:].iterrows():
        previous_position = float(current[-1][axis])
        current_position = float(row[axis])

        if abs(current_position - previous_position) <= 8:
            current.append(row)
        else:
            clusters.append(current)
            current = [row]

    clusters.append(current)

    for i, cluster in enumerate(clusters, start=1):
        c = pd.DataFrame(cluster)

        records.append({
            "Margin": margin,
            "Candidate_Number": i,
            "Detection_Count": len(c),
            "Mean_Centre_X": round(c["Centre_X"].mean(), 2),
            "Mean_Centre_Y": round(c["Centre_Y"].mean(), 2),
            "Position_Spread_px": round(c[axis].max()-c[axis].min(), 2),
            "Maximum_Segment_Length_px": round(c["Length_px"].max(), 2),
            "Coordinate_Assignment": "PENDING_VISUAL_VERIFICATION",
            "GCP_Status": "NOT_ACCEPTED"
        })

summary = pd.DataFrame(records)

expected = {
    "LEFT": 2,
    "RIGHT": 2,
    "TOP": 3,
    "BOTTOM": 3
}

print("\nConsolidated candidate clusters:", len(summary))
print("\nCandidate counts versus expected:")
for margin, count in expected.items():
    actual = int((summary["Margin"] == margin).sum())
    print(f"{margin}: {actual} candidates; expected {count}")

display(summary)

csv_out = OUT / "Step7ID4G12_Consolidated_Tick_Candidates.csv"
xlsx_out = OUT / "Step7ID4G12_Consolidated_Tick_Candidates.xlsx"

summary.to_csv(csv_out, index=False)
summary.to_excel(xlsx_out, index=False)

print("\nSaved:")
print(csv_out)
print(xlsx_out)
print("\nAccepted GCPs: 0")


**[orig #320]** Match ticks to the printed labels; update the master graticule register  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

candidate_file = OUT / "Step7ID4G12_Consolidated_Tick_Candidates.csv"
master_file = OUT / "Step7ID4G2_Graticule_Coordinate_Register.csv"

assert candidate_file.exists(), "Consolidated tick candidates missing."
assert master_file.exists(), "Master coordinate register missing."

candidates = pd.read_csv(candidate_file)
master = pd.read_csv(master_file)

# Match candidates in their spatial order to the printed labels.
assignments = {
    ("LEFT", 1): ("L2", 28.0),
    ("LEFT", 2): ("L3", 27.5),
    ("RIGHT", 1): ("R2", 28.0),
    ("RIGHT", 2): ("R3", 27.5),
    ("TOP", 1): ("T2", 76.0),
    ("TOP", 2): ("T3", 76.5),
    ("TOP", 3): ("T4", 77.0),
    ("BOTTOM", 1): ("B2", 76.0),
    ("BOTTOM", 2): ("B3", 76.5),
    ("BOTTOM", 3): ("B4", 77.0)
}

measured = []

for _, row in candidates.iterrows():
    key = (row["Margin"], int(row["Candidate_Number"]))

    if key not in assignments:
        continue

    point_id, coordinate_dd = assignments[key]

    measured.append({
        "Point_ID": point_id,
        "Coordinate_DD": coordinate_dd,
        "Pixel_X": row["Mean_Centre_X"],
        "Pixel_Y": row["Mean_Centre_Y"],
        "Detection_Count": row["Detection_Count"],
        "Position_Spread_px": row["Position_Spread_px"],
        "Pixel_Status": "AUTOMATED_CANDIDATE_VISUALLY_REVIEWED",
        "GCP_Status": "PENDING_TRANSFORMATION_VALIDATION"
    })

measured_df = pd.DataFrame(measured)

# Update existing master register without changing coordinate transcription.
for col in ["Pixel_X", "Pixel_Y"]:
    if col not in master.columns:
        master[col] = np.nan

for col in ["Pixel_Position_Status", "Coordinate_Status"]:
    if col not in master.columns:
        master[col] = "PENDING"

for _, row in measured_df.iterrows():
    mask = master["Point_ID"] == row["Point_ID"]

    if mask.sum() != 1:
        print("WARNING: Point ID not unique or missing:", row["Point_ID"])
        continue

    master.loc[mask, "Pixel_X"] = row["Pixel_X"]
    master.loc[mask, "Pixel_Y"] = row["Pixel_Y"]
    master.loc[mask, "Pixel_Position_Status"] = row["Pixel_Status"]
    master.loc[mask, "Coordinate_Status"] = row["GCP_Status"]

# Keep all corner pixel positions pending.
master.loc[
    master["Point_ID"].isin(["L1", "L4", "R1", "R4", "T1", "T5", "B1", "B5"]),
    "Pixel_Position_Status"
] = "PENDING_CORNER_PIXEL_MEASUREMENT"

master.to_csv(
    OUT / "Step7ID4G13_Master_Graticule_Register.csv",
    index=False
)
master.to_excel(
    OUT / "Step7ID4G13_Master_Graticule_Register.xlsx",
    index=False
)

# Opposite-margin pixel offset assessment
pairs = [
    ("L2", "R2", "Latitude 28 degrees"),
    ("L3", "R3", "Latitude 27 degrees 30 minutes"),
    ("T2", "B2", "Longitude 76 degrees"),
    ("T3", "B3", "Longitude 76 degrees 30 minutes"),
    ("T4", "B4", "Longitude 77 degrees")
]

offsets = []

for p1, p2, label in pairs:
    a = measured_df.loc[measured_df.Point_ID == p1].iloc[0]
    b = measured_df.loc[measured_df.Point_ID == p2].iloc[0]

    offsets.append({
        "Coordinate": label,
        "Point_1": p1,
        "Point_2": p2,
        "Delta_X_pixels": round(b.Pixel_X-a.Pixel_X, 2),
        "Delta_Y_pixels": round(b.Pixel_Y-a.Pixel_Y, 2),
        "Interpretation": "PIXEL_OFFSET_REQUIRES_PROJECTION_ASSESSMENT"
    })

offset_df = pd.DataFrame(offsets)

offset_df.to_csv(
    OUT / "Step7ID4G13_Opposite_Margin_Pixel_Offsets.csv",
    index=False
)
offset_df.to_excel(
    OUT / "Step7ID4G13_Opposite_Margin_Pixel_Offsets.xlsx",
    index=False
)

print("Master register observations:", len(master))
print("Interior pixel positions populated:", len(measured_df))
print("Corner positions still pending.")
print("Accepted GCPs: 0")
print("\nUpdated interior register:")
display(measured_df)

print("\nOpposite-margin offsets:")
display(offset_df)

print("\nSaved outputs in:", OUT)


## 3. Embedded original image and working crop

**[orig #321]** Inspect PDF page 8 (Plate IV)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from pathlib import Path
import fitz
import json

PDF = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Alwar_Hydrogeological_Atlas_2013.pdf"
)

assert PDF.exists(), f"PDF not found: {PDF}"

doc = fitz.open(PDF)

print("PDF page count:", len(doc))
print("\nPDF metadata:")
print(json.dumps(doc.metadata, indent=2))

# The geological map was identified on PDF page 8 (zero-based index 7).
page_index = 7
page = doc[page_index]

print("\nGeological map page information:")
print("Page number:", page_index + 1)
print("Page dimensions:", page.rect)
print("Page rotation:", page.rotation)

print("\nText extracted from geological map page:")
print(page.get_text("text")[:8000])

print("\nEmbedded images:")
images = page.get_images(full=True)
print("Image count:", len(images))

for i, info in enumerate(images):
    xref = info[0]
    width = info[2]
    height = info[3]
    bpc = info[4]
    colorspace = info[5]

    print({
        "image_number": i + 1,
        "xref": xref,
        "width": width,
        "height": height,
        "bits_per_component": bpc,
        "colorspace": colorspace
    })

doc.close()


**[orig #322]** Upgrade PyMuPDF  
<sub>• output not saved in the source notebook</sub>

In [ ]:
!pip -q install --upgrade PyMuPDF

**[orig #323]** Check PyMuPDF  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import fitz

print("PyMuPDF version:", fitz.VersionBind)
print("Installation successful.")

**[orig #324]** Extract the embedded original image of Plate IV  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import fitz
from pathlib import Path
from PIL import Image
import io
import numpy as np
import matplotlib.pyplot as plt

PDF = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Alwar_Hydrogeological_Atlas_2013.pdf"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing/"
    "Step7ID4G15_Original_Embedded_Map"
)
OUT.mkdir(parents=True, exist_ok=True)

doc = fitz.open(PDF)
page = doc[7]

# Extract the largest embedded image, previously identified as XREF 81
images = page.get_images(full=True)
image_info = max(images, key=lambda x: x[2] * x[3])

xref = image_info[0]
pix = fitz.Pixmap(doc, xref)

if pix.n - pix.alpha > 3:
    pix = fitz.Pixmap(fitz.csRGB, pix)

image_bytes = pix.tobytes("png")
embedded = Image.open(io.BytesIO(image_bytes)).convert("RGB")

embedded_path = OUT / "Alwar_Plate_IV_Embedded_Original.png"
embedded.save(embedded_path)

print("Extracted XREF:", xref)
print("Embedded image dimensions:", embedded.size)
print("Saved:", embedded_path)

# Compare with our previously extracted 4210 x 2975 image
previous_path = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/Actual_Geological_Map/"
    "Alwar_Plate_IV_Geological_Map_HighRes.png"
)

previous = Image.open(previous_path).convert("RGB")

print("\nPrevious image dimensions:", previous.size)
print("Embedded image dimensions:", embedded.size)

# Display both without changing their saved originals
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(previous)
axes[0].set_title("Previously extracted map")
axes[0].axis("off")

axes[1].imshow(embedded)
axes[1].set_title("Original embedded PDF image")
axes[1].axis("off")

plt.tight_layout()
plt.show()

doc.close()


**[orig #325]** Save the working crop and image metadata  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path
from PIL import Image
import json

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence"
)

SOURCE = (
    BASE / "Official_Geological_Maps" /
    "Actual_Geological_Map" /
    "Alwar_Plate_IV_Geological_Map_HighRes.png"
)

OUT = (
    BASE / "Step7ID4G_Graticule_Georeferencing" /
    "Step7ID4G16_Plate_IV_Working_Image"
)
OUT.mkdir(parents=True, exist_ok=True)

img = Image.open(SOURCE).convert("RGB")
W, H = img.size

# Previously identified inner Plate IV map neatline.
# These are candidate boundaries, not yet final GCPs.
x0, y0 = 413, 317
x1, y1 = 2019, 1761

# Save a crop with a small external margin for inspection.
margin = 15
crop_box = (
    max(0, x0-margin),
    max(0, y0-margin),
    min(W, x1+margin),
    min(H, y1+margin)
)

plate = img.crop(crop_box)

crop_path = OUT / "Alwar_Plate_IV_Working_Crop.png"
plate.save(crop_path)

# Save exact pixel-coordinate metadata.
metadata = {
    "source_image": str(SOURCE),
    "source_width": W,
    "source_height": H,
    "candidate_neatline": [x0, y0, x1, y1],
    "crop_box": list(crop_box),
    "crop_width": plate.width,
    "crop_height": plate.height,
    "status": "PROVISIONAL_NEATLINE_CROP_NOT_GEOREFERENCED"
}

with open(OUT / "Step7ID4G16_Image_Metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Source dimensions:", img.size)
print("Working crop dimensions:", plate.size)
print("Saved:", crop_path)

display(plate.resize((900, int(900*plate.height/plate.width))))

**[orig #326]** Convert graticule points to working-crop pixel coordinates  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw
import numpy as np

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence"
)

GEO = BASE / "Step7ID4G_Graticule_Georeferencing"
IN_CSV = GEO / "Step7ID4G13_Master_Graticule_Register.csv"

WORK = GEO / "Step7ID4G16_Plate_IV_Working_Image"
IMG_PATH = WORK / "Alwar_Plate_IV_Working_Crop.png"

OUT = GEO / "Step7ID4G17_Crop_Graticule_Verification"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(IN_CSV)
img = Image.open(IMG_PATH).convert("RGB")

print("Register columns:", df.columns.tolist())

# Identify pixel coordinate columns.
def find_col(options, columns):
    lookup = {str(c).lower().replace(" ", "_"): c for c in columns}
    for option in options:
        if option in lookup:
            return lookup[option]
    return None

xcol = find_col(
    ["pixel_x", "pixel_x_original", "x_pixel", "mean_x"],
    df.columns
)
ycol = find_col(
    ["pixel_y", "pixel_y_original", "y_pixel", "mean_y"],
    df.columns
)

if xcol is None or ycol is None:
    raise ValueError(
        "Could not identify pixel columns. Inspect the printed column names."
    )

# Retain original positions; derive working-crop positions.
df["Original_Pixel_X"] = pd.to_numeric(df[xcol], errors="coerce")
df["Original_Pixel_Y"] = pd.to_numeric(df[ycol], errors="coerce")

CROP_X_OFFSET = 398
CROP_Y_OFFSET = 302

df["Crop_Pixel_X"] = df["Original_Pixel_X"] - CROP_X_OFFSET
df["Crop_Pixel_Y"] = df["Original_Pixel_Y"] - CROP_Y_OFFSET

# Flag points that fall inside the working image.
df["Inside_Working_Crop"] = (
    df["Crop_Pixel_X"].between(0, img.width - 1) &
    df["Crop_Pixel_Y"].between(0, img.height - 1)
)

# Save the transferred register.
csv_out = OUT / "Step7ID4G17_Crop_Graticule_Register.csv"
xlsx_out = OUT / "Step7ID4G17_Crop_Graticule_Register.xlsx"

df.to_csv(csv_out, index=False)
df.to_excel(xlsx_out, index=False)

# Create verification overlay.
overlay = img.copy()
draw = ImageDraw.Draw(overlay)

for _, row in df.iterrows():
    x, y = row["Crop_Pixel_X"], row["Crop_Pixel_Y"]

    if pd.notna(x) and pd.notna(y) and row["Inside_Working_Crop"]:
        x, y = int(round(x)), int(round(y))
        draw.ellipse((x-7, y-7, x+7, y+7), outline="red", width=3)

        # Use an available identifier column if present.
        label_col = find_col(
            ["observation_id", "point_id", "control_point_id", "label"],
            df.columns
        )
        label = str(row[label_col]) if label_col else str(row.name)
        draw.text((x+9, y-12), label, fill="red")

overlay_path = OUT / "Step7ID4G17_Graticule_Crop_Overlay.png"
overlay.save(overlay_path)

print("\nWorking image:", img.size)
print("Register rows:", len(df))
print("Points with pixel coordinates:",
      df[["Original_Pixel_X", "Original_Pixel_Y"]].notna().all(axis=1).sum())
print("Points inside crop:", df["Inside_Working_Crop"].sum())
print("Saved:", csv_out)
print("Saved:", xlsx_out)
print("Saved:", overlay_path)

display(overlay.resize((800, int(800 * overlay.height / overlay.width))))

**[orig #327]** Check the graticule registers  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path
import pandas as pd

GEO = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

files = [
    GEO / "Step7ID4G13_Master_Graticule_Register.csv",
    GEO / "Step7ID4G17_Crop_Graticule_Verification" /
         "Step7ID4G17_Crop_Graticule_Register.csv"
]

for f in files:
    print("\n" + "="*75)
    print("FILE:", f.name)

    if not f.exists():
        print("MISSING")
        continue

    d = pd.read_csv(f)
    print("Shape:", d.shape)
    print("Columns:", d.columns.tolist())
    print(d.to_string(index=False))

## 4. Edge geometry and affine fit

**[orig #328]** Edge geometry from the ten reviewed interior ticks  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

GEO = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

CSV = GEO / "Step7ID4G17_Crop_Graticule_Verification" / \
      "Step7ID4G17_Crop_Graticule_Register.csv"

OUT = GEO / "Step7ID4G19_Edge_Geometry_Audit"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV)

# Only use the ten visually reviewed interior ticks.
d = df[
    df["Pixel_Position_Status"].astype(str)
      .str.contains("VISUALLY_REVIEWED", na=False)
].copy()

d["Coordinate_Decimal"] = pd.to_numeric(
    d["Coordinate_Decimal"], errors="coerce"
)
d["Crop_Pixel_X"] = pd.to_numeric(
    d["Crop_Pixel_X"], errors="coerce"
)
d["Crop_Pixel_Y"] = pd.to_numeric(
    d["Crop_Pixel_Y"], errors="coerce"
)

results = []

for margin, g in d.groupby("Map_Margin"):

    if margin in ["TOP", "BOTTOM"]:
        x = g["Crop_Pixel_X"].to_numpy()
        coord = g["Coordinate_Decimal"].to_numpy()
        axis = "X"
        coordinate_name = "Longitude"

    else:
        x = g["Crop_Pixel_Y"].to_numpy()
        coord = g["Coordinate_Decimal"].to_numpy()
        axis = "Y"
        coordinate_name = "Latitude"

    order = np.argsort(x)
    x = x[order]
    coord = coord[order]

    if len(x) >= 2:
        slope, intercept = np.polyfit(x, coord, 1)
        predicted = slope * x + intercept
        residual = coord - predicted

        rmse = float(np.sqrt(np.mean(residual**2)))
        max_abs = float(np.max(np.abs(residual)))

        results.append({
            "Margin": margin,
            "Coordinate": coordinate_name,
            "Pixel_Axis": axis,
            "Number_Of_Ticks": len(x),
            "Pixel_Min": x.min(),
            "Pixel_Max": x.max(),
            "Coordinate_Min": coord.min(),
            "Coordinate_Max": coord.max(),
            "Slope_Degrees_Per_Pixel": slope,
            "Intercept": intercept,
            "Linear_Fit_RMSE_Degrees": rmse,
            "Maximum_Absolute_Residual_Degrees": max_abs,
            "Status": "DIAGNOSTIC_ONLY"
        })

summary = pd.DataFrame(results)

print("\nEDGE GEOMETRY SUMMARY")
print(summary.to_string(index=False))

summary.to_csv(
    OUT / "Step7ID4G19_Edge_Geometry_Summary.csv",
    index=False
)

# Pairwise top/bottom and left/right tick offsets.
pairs = [
    ("T2", "B2"),
    ("T3", "B3"),
    ("T4", "B4"),
    ("L2", "R2"),
    ("L3", "R3")
]

offsets = []

for a, b in pairs:
    A = d[d["Point_ID"] == a]
    B = d[d["Point_ID"] == b]

    if len(A) == 1 and len(B) == 1:
        offsets.append({
            "Point_A": a,
            "Point_B": b,
            "Delta_X_Crop_px":
                float(B.iloc[0]["Crop_Pixel_X"] -
                      A.iloc[0]["Crop_Pixel_X"]),
            "Delta_Y_Crop_px":
                float(B.iloc[0]["Crop_Pixel_Y"] -
                      A.iloc[0]["Crop_Pixel_Y"])
        })

offset_df = pd.DataFrame(offsets)

print("\nOPPOSITE-MARGIN OFFSETS")
print(offset_df.to_string(index=False))

offset_df.to_csv(
    OUT / "Step7ID4G19_Opposite_Margin_Offsets.csv",
    index=False
)

print("\nSaved audit files in:", OUT)

**[orig #329]** **MAIN** Affine fit of the graticule (residuals in metres)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

GEO = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Step7ID4G_Graticule_Georeferencing"
)

CSV = (
    GEO / "Step7ID4G17_Crop_Graticule_Verification" /
    "Step7ID4G17_Crop_Graticule_Register.csv"
)

OUT = GEO / "Step7ID4G20_Affine_Diagnostic"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV)

d = df[
    df["Pixel_Position_Status"].astype(str)
      .str.contains("VISUALLY_REVIEWED", na=False)
].copy()

for col in ["Crop_Pixel_X", "Crop_Pixel_Y", "Coordinate_Decimal"]:
    d[col] = pd.to_numeric(d[col], errors="coerce")

d = d.dropna(subset=[
    "Crop_Pixel_X", "Crop_Pixel_Y", "Coordinate_Decimal"
])

# Each observation supplies either longitude or latitude.
A_rows = []
values = []
metadata = []

for _, r in d.iterrows():

    x = r["Crop_Pixel_X"]
    y = r["Crop_Pixel_Y"]
    val = r["Coordinate_Decimal"]

    if r["Coordinate_Type"] == "LONGITUDE":
        A_rows.append([1, x, y, 0, 0, 0])
        metadata.append((r["Point_ID"], "LONGITUDE", val))

    elif r["Coordinate_Type"] == "LATITUDE":
        A_rows.append([0, 0, 0, 1, x, y])
        metadata.append((r["Point_ID"], "LATITUDE", val))

    values.append(val)

A = np.asarray(A_rows, dtype=float)
values = np.asarray(values, dtype=float)

coef, residuals, rank, singular = np.linalg.lstsq(
    A, values, rcond=None
)

pred = A @ coef
errors = values - pred

result = pd.DataFrame(metadata, columns=[
    "Point_ID", "Coordinate_Type", "Observed_Degrees"
])

result["Predicted_Degrees"] = pred
result["Residual_Degrees"] = errors

# Approximate residual in metres, using local scale.
result["Residual_Approx_Metres"] = np.where(
    result["Coordinate_Type"] == "LATITUDE",
    errors * 111320,
    errors * 111320 * np.cos(np.deg2rad(27.5))
)

print("Number of scalar observations:", len(values))
print("Number of fitted coefficients:", len(coef))
print("Matrix rank:", rank)

print("\nAFFINE COEFFICIENTS")
print("Longitude = a0 + a1*X + a2*Y")
print("Latitude  = b0 + b1*X + b2*Y")

print("a0, a1, a2:", coef[:3])
print("b0, b1, b2:", coef[3:])

print("\nPOINT RESIDUALS")
print(result.to_string(index=False))

print("\nOVERALL DIAGNOSTICS")
print("RMSE degrees:", np.sqrt(np.mean(errors**2)))
print("Maximum absolute residual degrees:", np.max(np.abs(errors)))
print(
    "Maximum approximate residual metres:",
    result["Residual_Approx_Metres"].abs().max()
)

print("\nIMPORTANT: DIAGNOSTIC FIT ONLY. NOT AN ACCEPTED CRS OR GCP TRANSFORMATION.")

result.to_csv(
    OUT / "Step7ID4G20_Affine_Point_Residuals.csv",
    index=False
)

pd.DataFrame({
    "Coefficient": [
        "a0_longitude", "a1_longitude_X", "a2_longitude_Y",
        "b0_latitude", "b1_latitude_X", "b2_latitude_Y"
    ],
    "Value": coef
}).to_csv(
    OUT / "Step7ID4G20_Affine_Coefficients.csv",
    index=False
)

print("\nSaved:", OUT)